In [1]:
import sys
sys.path.append("..")
import pandas as pd

df = pd.read_csv("../data/urls.csv")
df.columns = ["url", "label"]
df["label"] = df["label"].map({"good": 0, "bad": 1})
print(df.shape)
print(df["label"].value_counts())
df.head()

(549346, 2)
label
0    392924
1    156422
Name: count, dtype: int64


,url,label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,1
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,1
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,1
3,mail.printakid.com/www.online.americanexpress....,1
4,thewhiskeydregs.com/wp-content/themes/widescre...,1


In [2]:
from ml.features import extract_features

df_small = df.sample(50000, random_state=42)
X = pd.DataFrame([extract_features(u) for u in df_small["url"]])
y = df_small["label"].values
X.head()

,url_length,host_length,path_length,num_dots,num_hyphens,num_digits,digit_ratio,num_at,num_question,num_equals,num_percent,num_slashes,num_subdomains,has_ip,has_suspicious_word,is_shortener
0,30,14,8,1,0,0,0.000000,0,1,1,0,2,0,0,0,0
1,76,14,52,1,0,11,0.144737,0,1,1,3,3,0,0,0,0
2,64,9,55,2,0,15,0.234375,0,0,0,0,4,0,0,0,0
3,46,21,15,2,0,0,0.000000,0,1,1,0,1,0,0,0,0
4,371,45,34,3,6,42,0.113208,0,1,3,3,3,1,0,1,0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0       0.90      0.95      0.92      7144
           1       0.84      0.74      0.79      2856

    accuracy                           0.89     10000
   macro avg       0.87      0.84      0.85     10000
weighted avg       0.88      0.89      0.88     10000

[[6752  392]
 [ 750 2106]]


In [4]:
pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

path_length            0.146727
url_length             0.125399
host_length            0.114402
digit_ratio            0.099669
num_digits             0.094526
num_slashes            0.093619
num_dots               0.090403
has_suspicious_word    0.075051
num_hyphens            0.051246
num_subdomains         0.037525
num_equals             0.020808
num_question           0.019574
has_ip                 0.013675
is_shortener           0.009574
num_percent            0.005081
num_at                 0.002720
dtype: float64

In [5]:
tests = [
    "google.com/search?q=python",
    "github.com/openai",
    "docs.google.com/document/d/1aBcDeFgHiJkLmNoPqRsTuVwXyZ0123456789/edit?usp=sharing",
    "amazon.in/dp/B09XYZ1234/ref=sr_1_1?keywords=laptop&qid=1699999999",
    "secure-login-sbi-kyc-update.xyz/verify",
    "paypal.com.cgi.bin.webscr.login.submit.example.com/login.php",
    "sbi-kyc-update-now.top",
]
Xt = pd.DataFrame([extract_features(u) for u in tests])
for u, p in zip(tests, model.predict_proba(Xt)[:, 1]):
    print(f"{p:.2f}  {u}")

0.18  google.com/search?q=python
0.34  github.com/openai
0.33  docs.google.com/document/d/1aBcDeFgHiJkLmNoPqRsTuVwXyZ0123456789/edit?usp=sharing
0.46  amazon.in/dp/B09XYZ1234/ref=sr_1_1?keywords=laptop&qid=1699999999
0.51  secure-login-sbi-kyc-update.xyz/verify
0.98  paypal.com.cgi.bin.webscr.login.submit.example.com/login.php
0.64  sbi-kyc-update-now.top


In [6]:
import joblib
joblib.dump(model, "../ml/model.joblib")

['../ml/model.joblib']

In [7]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

def clean_url(u):
    return re.sub(r"^https?://", "", str(u).strip().lower())

d = df.copy()
d["clean"] = d["url"].map(clean_url)
d = d.drop_duplicates(subset="clean")
print("after removing duplicates:", d.shape)
d = d.sample(100000, random_state=42)

Xtr_txt, Xte_txt, ytr, yte = train_test_split(
    d["clean"], d["label"], test_size=0.2, stratify=d["label"], random_state=42)

vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5),
                      max_features=200000, sublinear_tf=True)
Xtr = vec.fit_transform(Xtr_txt)
Xte = vec.transform(Xte_txt)

clf = LogisticRegression(max_iter=1000, C=5, class_weight="balanced")
clf.fit(Xtr, ytr)

pred = clf.predict(Xte)
print(classification_report(yte, pred))
print(confusion_matrix(yte, pred))

after removing duplicates: (507108, 3)
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     15529
           1       0.93      0.92      0.93      4471

    accuracy                           0.97     20000
   macro avg       0.95      0.95      0.95     20000
weighted avg       0.97      0.97      0.97     20000

[[15221   308]
 [  343  4128]]


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

           0       0.90      0.95      0.92      7144
           1       0.84      0.74      0.79      2856

    accuracy                           0.89     10000
   macro avg       0.87      0.84      0.85     10000
weighted avg       0.88      0.89      0.88     10000

[[6752  392]
 [ 750 2106]]


In [8]:
probs = clf.predict_proba(vec.transform([clean_url(u) for u in tests]))[:, 1]
for u, p in zip(tests, probs):
    print(f"{p:.2f}  {u}")

0.01  google.com/search?q=python
0.06  github.com/openai
0.98  docs.google.com/document/d/1aBcDeFgHiJkLmNoPqRsTuVwXyZ0123456789/edit?usp=sharing
0.21  amazon.in/dp/B09XYZ1234/ref=sr_1_1?keywords=laptop&qid=1699999999
1.00  secure-login-sbi-kyc-update.xyz/verify
1.00  paypal.com.cgi.bin.webscr.login.submit.example.com/login.php
0.99  sbi-kyc-update-now.top


In [9]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

def clean_url(u):
    return re.sub(r"^https?://", "", str(u).strip().lower())

d = df.copy()
d["clean"] = d["url"].map(clean_url)
d = d.drop_duplicates(subset="clean")
print("after removing duplicates:", d.shape)
d = d.sample(100000, random_state=42)

Xtr_txt, Xte_txt, ytr, yte = train_test_split(
    d["clean"], d["label"], test_size=0.2, stratify=d["label"], random_state=42)

vec = TfidfVectorizer(analyzer="char", ngram_range=(3, 5),
                      max_features=200000, sublinear_tf=True)
Xtr = vec.fit_transform(Xtr_txt)
Xte = vec.transform(Xte_txt)

clf = LogisticRegression(max_iter=1000, C=5, class_weight="balanced")
clf.fit(Xtr, ytr)

pred = clf.predict(Xte)
print(classification_report(yte, pred))
print(confusion_matrix(yte, pred))

after removing duplicates: (507108, 3)
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     15529
           1       0.93      0.92      0.93      4471

    accuracy                           0.97     20000
   macro avg       0.95      0.95      0.95     20000
weighted avg       0.97      0.97      0.97     20000

[[15221   308]
 [  343  4128]]


In [10]:
probs = clf.predict_proba(vec.transform([clean_url(u) for u in tests]))[:, 1]
for u, p in zip(tests, probs):
    print(f"{p:.2f}  {u}")

0.01  google.com/search?q=python
0.06  github.com/openai
0.98  docs.google.com/document/d/1aBcDeFgHiJkLmNoPqRsTuVwXyZ0123456789/edit?usp=sharing
0.21  amazon.in/dp/B09XYZ1234/ref=sr_1_1?keywords=laptop&qid=1699999999
1.00  secure-login-sbi-kyc-update.xyz/verify
1.00  paypal.com.cgi.bin.webscr.login.submit.example.com/login.php
0.99  sbi-kyc-update-now.top


In [11]:
import joblib
joblib.dump({"vec": vec, "clf": clf}, "../ml/url_model.joblib")

['../ml/url_model.joblib']